In [1]:
from echo.settings import debug_mode
from echo.indexing import create_tables
from echo.runner import make_call
import nest_asyncio
import asyncio


nest_asyncio.apply()

seller = 'https://whatfix.com/'
buyer = 'https://www.manpowergroup.com'

create_tables(seller)


debug_mode

Created tables for https://whatfix.com/


False

In [2]:
from echo.step_templates.generic import CallType

call_id = 1
inputs = {
    'seller': seller,
    'call_id': call_id,
    'stakeholders': [
        "Product Manager",
        "Chief Financial Officer",
        "Chief Technology Officer",
        "VP of Sales"
    ]
}

discovery_data = asyncio.run(make_call(call_type=CallType.DISCOVERY.value, clients=[buyer], inputs=inputs))

Making discovery call for ['https://www.manpowergroup.com']
Getting analysis Data
Number of Clients:  1


Getting Data: 100%|██████████| 1/1 [00:00<00:00, 219.64it/s]

Getting Data for https://www.manpowergroup.com
Seller Research Data Found
Found Seller: https://whatfix.com/ Website Content
Buyer Research Data Found


In [3]:
#inputs['call_id'] = call_id + 1
#demo_data = asyncio.run(make_call(call_type=CallType.DEMO.value, clients=[buyer], inputs=inputs))

In [4]:
#inputs['call_id'] = call_id + 2
#pricing_data = asyncio.run(make_call(call_type=CallType.PRICING.value, clients=[buyer], inputs=inputs))

In [5]:
#inputs['call_id'] = call_id + 3
#negotiation_data = asyncio.run(make_call(call_type=CallType.NEGOTIATION.value, clients=[buyer], inputs=inputs))

In [ ]:
from echo.data.indexes import IndexType, IndexDataType
from echo.query_executor import Query, LlamaSubQuery, QueryChain
from echo.step_templates.utilities.account_plan_creation import QueryTypes
from echo.query_executor import arun_query_chain
# from echo.query_executor import aget_query_response
import asyncio

import nest_asyncio
nest_asyncio.apply()


inputs = {
    "seller": seller,
    "buyer": buyer,
}

In [7]:
account_plan = Query(
    query="You're a strategic B2B seller. Based on the following public signals about {buyer}.\n" 
    "Extract 3-5 key initiatives or priorities the company is likely pursuing this year or quarter.\n"
    "Phrase each as a business goal. Do NOT include vague goals. Be specific.\n"
    "Signals for the various initiatives are given below:\n"

    "\nReturn format:\n"
    "- Initiative: clear description\n"
    "- Supporting evidence: source\n",
    sub_queries=[
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors for the account that that client needs to consider?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news for the account?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        )
    ],
    output_name="account_plan"
)

In [8]:
# response = asyncio.run(aget_query_response(echo_query=account_plan, inputs=inputs))
# print(response[0])

In [9]:
account_plan_value_prop = Query(
    query="""
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyeraccount?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value},
        )
    ],
    output_name="account_plan_value_prop",
)

In [10]:
# response = asyncio.run(aget_query_response(echo_query=account_plan_value_prop, inputs=inputs))
# print(response[0])

In [11]:
competitor_differentiator_value_prop = Query(
    query="""You need to generate a few value propositions and business cases that the seller Whatfix can then use to sell their solution to the buyer Manpower group. 
            The seller is whatfix and the buyer is manpower group
            First identify the top financial, strategic, competitive and priorities evident from news and media to craft top issues and focus points of the buyer.
            Next deeply understand the sellers product, the core problems it has and does solve for its buyers.
            Please make sure to properly align value prop to actual business cases and not just generic value prop. 
            Also understand deeply what the seller sells and the kind of impact it can have before answering. 
            Think deeply
            Now finally, craft a set of value propositions and business cases that the sellers product can solve in alignment with the buyers priorities identified. This will be used by an account executive to pitch the product to the buyer and align with their priorities. so be clear, detailed and specific.
            Use the sellers product info, testimonials, broad initiatives theyve tackled for other customers and how they can align with the buyers strategic, financial and competitive priorities. Also include news and media about the buyer into consideration for further hints and signals on buyer priorities
            """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyeraccount?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
)

In [12]:
# response = asyncio.run(aget_query_response(echo_query=competitor_differentiator_value_prop, inputs=inputs))
# print(response[0])

In [13]:
# 1) multi threading team gen 
from echo.query_executor import PerplexicaSourceExtraction, PerplexicaSubQuery


multi_threading_team_gen = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, for the ones marked relevant to the seller's product, "
        "Infer which internal team likely owns or sponsors each initiative. \n"
        "If multiple teams are involved, note primary and secondary.\n"

        "Input:\n"
        "{account_plan_value_prop}\n"

        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Reasoning:\n"
    ),
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_team_gen",
)

perplexity_search_query = Query(
    query="""
    You are provided with the company initiatives and the owning team for each of them.
    
    {multi_threading_team_gen}.
    
    You are provided with the company initiatives and the relevant information about from the web about the potential team members and employees of the company that would be relevant to the company initiatives.
    Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.
    """,
    sub_queries=[
        PerplexicaSubQuery(
            query=(
                "You are provided with the company initiatives and the owning team for each of them\n"
                "{multi_threading_team_gen}\n"
                "Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.\n"
            ),
            source_extraction_prompts=PerplexicaSourceExtraction(
                system_prompt=(
                    "You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning "
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                ),
                user_prompt=(
                    "You are provided with the company initiatives and the owning team for each of them\n"
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                    "Extract out the team members or employees from the below data\n"
                )
            )
        )
    ],
    output_name="perplexity_search_query"
)

# 2) do a perplexica serach here using team name and buyer name FOR EACH INITIATIVE RETURNED FROM ABOVE
# Search for all possible employees and leaders  from linkedin who belong to that team and extract role, name, and background


# 3) multi threading ROLE AND PERSON EXTRACTOR - use above response also aas input below additionally 

multi_threading_person_extractor = Query(
    query="""
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing.
        
        
        You are provided with the company initiatives and the owning team for each of them.
        {multi_threading_team_gen}.
        
        You are further provided with the relevant team members and employees of the company that would be relevant to the company initiatives as crawled from the web.
        {perplexity_search_query}
        
        Do the following:

        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority, team fit, and tenure. Prioritize those with likely budget/influence.

        Also, Classify each as a champion, decision maker, gatekeeper and influncer within the team responsible for the initiative.
        champion - one who directly owns the pain and will want it solved
        decision maker- the one with power to purchase in the team and for the initiative
        gatekeeper - the one who will block the deal from happening or be tough to convince. This is the only role that could be outside the team like procurement , legal etc.
        influencer - the one who will influence the decision maker and champion to buy the product.

        Return:
        - initiative
        - Name
        - Title
        - Tenure
        - Team
        - Reason they likely own this initiative
        - Classification (champion, decision maker, gatekeeper, influencer) and why

        here is the list of initiatives and the owning team for each of them:
        {multi_threading_team_gen}
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value} ## Has demo data separately
        )
    ],
    output_name="multi_threading_person_extractor"
)


# 4) multi threading outreach generator

multi_threading_outreach_generator = Query(
    query="""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer's title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {multi_threading_person_extractor}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)


In [14]:
# response = asyncio.run(aget_query_response(echo_query=multi_threading_team_gen, inputs=inputs))
# print(response[0])

In [15]:
query_chain = QueryChain(
    queries=[
        account_plan_value_prop,
        multi_threading_team_gen,
        perplexity_search_query,
        multi_threading_person_extractor,
        multi_threading_outreach_generator
    ]
)

In [ ]:
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

Running query 
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    
Running sub query What is the details on the industry and products of the buyer account?
Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'c

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': 'Relevant Context:\nThe buyer account, ManpowerGroup, operates in the Human Resource Services industry and is classified as an Enterprise in terms of company size. Their goals include transforming workforce strategies, delivering innovative solutions for global businesses, and ensuring adaptability in the face of workforce changes. Some of their use cases involve talent creation at scale, employment outlook insights, right management, and recruitment processing outsourcing. ManpowerGroup faces challenges such as global talent shortages, the need for adaptability, and integrating sustainability and governance into their workforce strategies. Their stakeholders include HR and business leaders, recruitment professionals, talent developme

Overriding of current TracerProvider is not allowed


Record does not exist, making API call...
No data extracted from sources.
No data extracted from sources.
Sub query context {'query': "You are provided with the company initiatives and the owning team for each of them\n**Inference of Internal Teams for Relevant Initiatives**\n==============================================\n\nBy analyzing ManpowerGroup's strategic initiatives and their relevance to Whatfix's product, we can infer the likely internal teams that own or sponsor each initiative. This will help the sales agent understand the key stakeholders and priorities within ManpowerGroup, increasing the chances of a successful sale.\n\n### 1. Talent Shortage Solutions\n#### Initiative: Talent Shortage Solutions\n#### Likely owning team(s): Recruitment Team, HR Team\n#### Reasoning: The initiative focuses on streamlining recruitment processes, which is a primary concern for the Recruitment Team and HR Team. Whatfix's Digital Adoption Platform can assist in this process, making it a rele

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': 'Relevant Context:\nThe buyer account belongs to ManpowerGroup, a company in the Human Resource Services industry with an Enterprise company size. ManpowerGroup focuses on workforce solutions across the talent lifecycle, from talent attraction to upskilling and reskilling. Their goals include transforming workforce strategies, delivering innovative solutions for global businesses, and future-proofing organizations. Some of their use cases are talent creation at scale, employment outlook insights, right management, and recruitment processing outsourcing. The challenges they face include global talent shortages, adaptability needs, and integrating sustainability and governance into their strategies. The stakeholders involved are HR and 

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'what are some companies the sellers product has helped before? be specific and metric drivenAnswer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': "Relevant Context:\nThe seller's product has helped companies like ManpowerGroup by optimizing CRM for sales enhancement, providing benefits in Human Capital Management (HCM), streamlining the recruitment process, modernizing legacy systems, aiding in digital transformation, and enhancing user adoption. Additionally, the product has assisted in user journey optimization, feature usage analysis, survey integration, digital transformation, AI adoption, change management, employee/user onboarding, training, and performance support."}
Running final query 
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participat

Exception while exporting Span batch.
Traceback (most recent call last):
  File "/Users/rap/echo/echo/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/Users/rap/echo/echo/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rap/echo/echo/.venv/lib/python3.12/site-packages/urllib3/connection.py", line 516, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.12/3.12.9/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1430, in getresponse
    response.begin()
  File "/opt/homebrew/Cellar/python@3.12/3.12.9/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_sta

In [17]:
for r in response.responses:
    print("Query: ", r.query)
    print("Response: ", r.response)

Query:  
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    
Response:  **Relevant Initiatives for Whatfix's Product**

### ManpowerGroup's Strategic Initiatives

#### 1. Talent Shortage Solutions
* "initiative": "Talent Shortage Solutions",
* "relevant": highly relevant,
* "mapped_to_product": "Recruitment Process Streamlining",
* "reasoning": "Whatfix's Digital Adoption Platform can assist Manpower

In [18]:
import echo.sqldb as sqldb

db = sqldb.get_records(seller, IndexType.SELLER_RESEARCH.value, condition_dict={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value})
print(type(db))
db[1]['data']['url']


<class 'list'>


'https://www.pendo.io'

In [ ]:
# endpoint 1 - for account plan tab
# returns two sets of data - account plan and value prop
from echo.query_executor import (
    ContextExtractionMode,
    Query,
    ResponseFormat,
    arun_queries,
)
from echo.queries import get_queries_multithreading, get_query_account_plan_query_chain  # type: ignore
from echo.queries import get_queries

# query_chain = QueryChain(
#     queries=[
#         account_plan,
#         account_plan_value_prop,
#     ]
# )
query_chain = get_query_account_plan_query_chain()
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

# send this to frontend
print(response.responses)


ImportError: cannot import name 'QueryTypes' from 'echo.tools.perplexity_search' (/Users/rap/echo/echo/echo_planner/echo/tools/perplexity_search.py)

In [ ]:
# endpoint 2 - for chat - keep same
# same logic as before

# just change function from aget_query_response to 
# 
# query_chain = QueryChain(
#     queries=[
#         query_instance_from_chat_as_before
#     ]
# )
# response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

# send this to frontend
# response.responses


In [22]:
# endpoint 3 - for multithreading tab


# 1) multi threading team gen 
from echo.query_executor import PerplexicaSourceExtraction, PerplexicaSubQuery
from echo.queries import get_multithread_query_chain # type: ignore

multi_threading_team_gen = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, for the ones marked relevant to the seller's product, "
        "Infer which internal team likely owns or sponsors each initiative. \n"
        "If multiple teams are involved, note primary and secondary.\n"

        "Input:\n"
        "{account_plan_value_prop}\n"

        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Reasoning:\n"
    ),
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_team_gen",
)

perplexity_search_query = Query(
    query="""
    You are provided with the company initiatives and the owning team for each of them.
    
    {multi_threading_team_gen}.
    
    You are provided with the company initiatives and the relevant information about from the web about the potential team members and employees of the company that would be relevant to the company initiatives.
    Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.
    """,
    sub_queries=[
        PerplexicaSubQuery(
            query=(
                "You are provided with the company initiatives and the owning team for each of them\n"
                "{multi_threading_team_gen}\n"
                "Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.\n"
            ),
            source_extraction_prompts=PerplexicaSourceExtraction(
                system_prompt=(
                    "You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning "
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                ),
                user_prompt=(
                    "You are provided with the company initiatives and the owning team for each of them\n"
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                    "Extract out the team members or employees from the below data\n"
                )
            )
        )
    ],
    output_name="perplexity_search_query"
)

# 2) do a perplexica serach here using team name and buyer name FOR EACH INITIATIVE RETURNED FROM ABOVE
# Search for all possible employees and leaders  from linkedin who belong to that team and extract role, name, and background


# 3) multi threading ROLE AND PERSON EXTRACTOR - use above response also aas input below additionally 

multi_threading_person_extractor = Query(
    query="""
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing.
        
        
        You are provided with the company initiatives and the owning team for each of them.
        {multi_threading_team_gen}.
        
        You are further provided with the relevant team members and employees of the company that would be relevant to the company initiatives as crawled from the web.
        {perplexity_search_query}
        
        Do the following:

        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority, team fit, and tenure. Prioritize those with likely budget/influence.

        Also, Classify each as a champion, decision maker, gatekeeper and influncer within the team responsible for the initiative.
        champion - one who directly owns the pain and will want it solved
        decision maker- the one with power to purchase in the team and for the initiative
        gatekeeper - the one who will block the deal from happening or be tough to convince. This is the only role that could be outside the team like procurement , legal etc.
        influencer - the one who will influence the decision maker and champion to buy the product.

        Return:
        - initiative
        - Name
        - Title
        - Tenure
        - Team
        - Reason they likely own this initiative
        - Classification (champion, decision maker, gatekeeper, influencer) and why

        here is the list of initiatives and the owning team for each of them:
        {multi_threading_team_gen}
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value} ## Has demo data separately
        )
    ],
    output_name="multi_threading_person_extractor"
)


# 4) multi threading outreach generator

multi_threading_outreach_generator = Query(
    query="""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer's title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {multi_threading_person_extractor}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)

query_chain = QueryChain(
    queries=[
        account_plan_value_prop,
        multi_threading_team_gen,
        perplexity_search_query,
        multi_threading_person_extractor,
        multi_threading_outreach_generator
    ]
)

#query_chain = get_multithread_query_chain(seller, buyer)
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

# send this to frontend
print(response.responses)

Running query 
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    
Running sub query What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com
Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answe

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers prioritiesAnswer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.comAnswer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': "Relevant Context:\nWhatfix provides a range of case studies and testimonials that align with the priorities of potential buyers like ManpowerGroup. The case studies and testimonials showcase how Whatfix's products have been instrumental in various scenarios such as CRM optimization, sales enhancement, human capital management benefits, recruitment process streamlining, legacy system modernization, digital transformation, user adoption, user journey optimization, feature usage analysis, sur

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': 'Relevant Context:\nThe buyer account belongs to ManpowerGroup, a company in the Human Resource Services industry with an Enterprise company size. ManpowerGroup focuses on workforce solutions across the talent lifecycle, from talent attraction to upskilling and reskilling. Their goals include transforming workforce strategies, delivering innovative solutions for global businesses, and future-proofing organizations. They offer services like Talent Creation at Scale, Right Management, and Recruitment Processing Outsourcing.\n\nAs for the seller, Whatfix provides digital adoption solutions aimed at enhancing organizational efficiency and performance. Their suite includes product analytics, interactive simulations for training, and suppor

...

Record does not exist, making API call...
No data extracted from sources.
No data extracted from sources.
Sub query context {'query': "You are provided with the company initiatives and the owning team for each of them\n**Initiative Relevance Analysis for ManpowerGroup**\n\n### 1. Talent Shortage Solutions\n- Initiative: Talent Shortage Solutions\n- Likely owning team(s): HR, Talent Acquisition\n- Reasoning: Whatfix's Digital Adoption Platform and Mirror product can help address talent shortages by providing interactive guidance and support for HR professionals, enabling them to efficiently manage talent acquisition and development processes. The platform's ability to provide personalized training and onboarding experiences can also help attract and retain top talent.\n\n### 2. Adaptability Measures\n- Initiative: Adaptability Measures\n- Likely owning team(s): IT, Learning and Development\n- Reasoning: Whatfix's Digital Adoption Platform can help organizations accelerate adaptability b

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?Answer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': 'Relevant Context:\nThe buyer account belongs to ManpowerGroup, a company in the Human Resource Services industry with an Enterprise company size. ManpowerGroup focuses on workforce solutions across the talent lifecycle, from talent attraction to upskilling and reskilling. Their goals include transforming workforce strategies, delivering innovative solutions for global businesses, and future-proofing organizations. They offer services such as talent creation at scale, employment outlook insights, right management, and more. The stakeholders involved are HR and business leaders, recruitment professionals, talent development professionals, and business owners.\n\nAs for the seller, Whatfix provides digital adoption solutions aimed at en

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'what are some companies the sellers product has helped before? be specific and metric drivenAnswer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': "Relevant Context:\nThe seller's product has helped companies like ManpowerGroup by optimizing CRM for sales enhancement, providing benefits in Human Capital Management (HCM), streamlining recruitment processes, modernizing legacy systems, facilitating digital transformation, enhancing user adoption, optimizing user journeys, analyzing feature usage, integrating surveys, supporting digital transformation and AI adoption, aiding in change management, and assisting in employee/user onboarding and training."}
Running final query 
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in

In [ ]:
# endpoint 4 - for competitor tab
# returns 3 queries
# fetch competitors
import echo.sqldb as sqldb
from echo.queries import get_competitor_query_chain # type: ignore

db = sqldb.get_records(seller, IndexType.SELLER_RESEARCH.value, condition_dict={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value})
competitors = []
for record in db:
    competitors.append(record['data']['url'])

competitors = list(set(competitors))
competitor_queries = []
for competitor in competitors:
    value_prop_query_competitor = Query(
                query=f"""You need to help an account executive of our seller company. {seller} differentiate against the competitor {competitor} in relevance to the buyer {buyer}.
                    The seller is trying to create a business case for the buyer and you need to help the account exectutive differentiate the sellers product from the competitors product.
                    The seller is {seller}, the competitor {competitor} and the buyer is {buyer}.
                    The top financial, strategic, competitive and priorities evident from news and media to craft top issues and focus points of the buyer are given.
                    Next deeply understand the sellers product, the core problems it solves for its buyers.
                    Next deeply understand the competitors product, the core porblems it solves for its buyers.
                    Consider deep differentiation and not just surface level differentiation between the seller and the COMPETITOR IN CONTEXT TO THE BUYERS PRIORITIES.
                    HIGHLIGHT WHERE THE SELLERS VALUE PROP CAN BE STRONGER AND WHY AND WHERE THE COMPETITORS VALUE PROP CAN BE STRONGER AND HOW TO TACKLE THAT IN THE BUSINESS CASE FOR THE BUYER.  
                    Please make sure to properly align value prop to actual business cases and not just generic value prop. 
                    Also understand deeply what the seller sells and the kind of impact it can have before answering. 
                    Think deeply
                    Now finally, craft a set of value propositions and business cases that the sellers product can solve in alignment with the buyers priorities identified. This will be used by an account executive to pitch the product to the buyer and align with their priorities. so be clear, detailed and specific.
                    Use the sellers product info, testimonials, broad initiatives theyve tackled for other customers and how they can align with the buyers strategic, financial and competitive priorities. 
                    Also include news and media about the buyer into consideration for further hints and signals on buyer priorities.
                    USE SAME DATA FOR COMPETITORS TO IDENTIFY HOW THE SELLER CAN TACKLE THE COMPETITORS VALUE PROP AND HOW THEY CAN ALIGN WITH THE BUYERS PRIORITIES BETTER.
                    """,
                sub_queries=[
                    LlamaSubQuery(
                        query="What is the details on the industry and products of the buyer account?",
                        index_type=IndexType.BUYER_RESEARCH,
                        inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
                    ),
            
                    LlamaSubQuery(
                        query="What are the top 3 financial priorities for the account to solve for?",
                        index_type=IndexType.BUYER_ACCOUNT_PLAN,
                        inputs={"query_type": QueryTypes.FMOD.value},
                    ),
                    LlamaSubQuery(
                        query="What are the top 3 competitors that buyer might be worried about and want to tackle",
                        index_type=IndexType.BUYER_ACCOUNT_PLAN,
                        inputs={"query_type": QueryTypes.COMPANALYSIS.value},
                    ),
                    LlamaSubQuery(
                        query="What is the most relevant news and recent media for the buyer account?",
                        index_type=IndexType.BUYER_ACCOUNT_PLAN,
                        inputs={"query_type": QueryTypes.RECENTNEWS.value},
                    ),
                    LlamaSubQuery(
                        query="What are the top 3 strategic priorities for the account to solve for?",
                        index_type=IndexType.BUYER_ACCOUNT_PLAN,
                        inputs={"query_type": QueryTypes.STRATEGY.value},
                    ),
                    LlamaSubQuery(
                        query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
                        index_type=IndexType.SELLER_RESEARCH,
                        # inputs={"query_type": QueryTypes.STRATEGY.value},
                    ),
                    LlamaSubQuery(
                        query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
                        index_type=IndexType.SELLER_RESEARCH,
                        inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
                    ),
                    LlamaSubQuery(
                        query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
                        index_type=IndexType.SELLER_RESEARCH,
                        inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}

                        # inputs={"query_type": QueryTypes.STRATEGY.value},
                    ),
                    LlamaSubQuery(
                        query=f"What are the top value propositions of the competitors {competitor} product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
                        index_type=IndexType.SELLER_RESEARCH,
                        inputs={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value}

                        # inputs={"seller":"https://www.pendo.io"}
                        # inputs={"query_type": QueryTypes.STRATEGY.value},
                    ),
                    LlamaSubQuery(
                        query=f"What are the exhaustive use cases and benefits of the competitor {competitor} product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
                        index_type=IndexType.SELLER_RESEARCH,
                        inputs={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value}
                    ),
                    LlamaSubQuery(
                        query=f"What case studies and testimonials do we have for the competitor {competitor} product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
                        index_type=IndexType.SELLER_RESEARCH,
                        inputs={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value}
                    ),
                ],
            )
    competitor_queries.append(value_prop_query_competitor)
query_chain = QueryChain(
    queries=competitor_queries,
)

query   _chain = get_competitor_query_chain(seller,buyer)
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

# send this to frontend
print(response.responses)


Running query You need to help an account executive of our seller company. https://whatfix.com/ differentiate against the competitor https://www.walkme.com in relevance to the buyer https://www.manpowergroup.com.
                        The seller is trying to create a business case for the buyer and you need to help the account exectutive differentiate the sellers product from the competitors product.
                        The seller is https://whatfix.com/, the competitor https://www.walkme.com and the buyer is https://www.manpowergroup.com.
                        The top financial, strategic, competitive and priorities evident from news and media to craft top issues and focus points of the buyer are given.
                        Next deeply understand the sellers product, the core problems it solves for its buyers.
                        Next deeply understand the competitors product, the core porblems it solves for its buyers.
                        Consider deep differentiat

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What case studies and testimonials do we have for the competitor https://www.walkme.com product? Please include details of the case studies and testimonials and how they align with the buyers prioritiesAnswer the question in context to the seller as https://whatfix.com/ selling their products to a potential buyer: https://www.manpowergroup.com', 'context': "Relevant Context:\nThe competitor website, WalkMe, showcases several case studies and testimonials that highlight the effectiveness of their Digital Adoption Platform (DAP) in enhancing user experiences and maximizing software investments. \n\nOne testimonial from Aaron Bloom, an Operations Associate at Practice Fusion, emphasizes the effectiveness of WalkMe in enhancing user understanding and feature utilization through on-demand tutorials and update alerts. This aligns with the buyer's priorities at ManpowerGroup, as it indicates that WalkMe's platform can improve user engagement and facilitate smoothe